In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F

# ==========================================
# 1. CẤU HÌNH THAM SỐ
# ==========================================
CONFIG = {
    "csv_path": "devset_images_gt.csv",
    "test_csv": "test.csv",
    "train_img_dir": "./devset_images/devset_images/", 
    "test_img_dir": "./testset_images/testset_images/",      
    "batch_size": 16,
    "epochs": 5,        # 5 epochs cho mỗi Fold
    "n_splits": 5,      # 5 Folds
    "lr": 2e-4,
    "img_size": 256,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

In [2]:
# ==========================================
# 2. DATASET VÀ AUGMENTATION
# ==========================================
class FloodDataset(Dataset):
    def __init__(self, df, img_dir, id_col, label_col=None, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.id_col = id_col
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, self.id_col]
        
        # Danh sách các đuôi ảnh có thể xuất hiện trong thư mục
        possible_exts = ['.jpg', '.png', '.gif', '.JPG', '.PNG', '.GIF']
        img_path = None
        
        # Vòng lặp tìm kiếm xem đuôi nào đang tồn tại
        for ext in possible_exts:
            temp_path = os.path.join(self.img_dir, f"{img_id}{ext}")
            if os.path.exists(temp_path):
                img_path = temp_path
                break # Tìm thấy thì dừng vòng lặp lại
                
        # Nếu duyệt qua hết các đuôi mà vẫn không thấy, báo lỗi đỏ để ta biết
        if img_path is None:
            raise FileNotFoundError(f"KHÔNG TÌM THẤY ảnh nào cho ID {img_id} trong thư mục {self.img_dir}")
        
        # Mở ảnh bằng đường dẫn đã tìm được
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        if self.label_col is not None:
            label = int(self.df.loc[idx, self.label_col])
            return image, torch.tensor(label, dtype=torch.float32)
        
        return image, img_id

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.6, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        # alpha = 0.6: Giống pos_weight, ưu tiên phạt nặng hơn nếu lỡ bỏ sót ảnh ngập lụt
        # gamma = 2.0: Hệ số tập trung, càng cao càng ép mô hình soi kỹ ảnh khó
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs là logits (chưa qua sigmoid)
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        
        # Tính xác suất dự đoán đúng (pt)
        pt = torch.exp(-bce_loss) 
        
        # Cốt lõi của Focal Loss: (1 - pt)^gamma
        # pt càng gần 1 (đoán càng đúng) -> (1-pt) tiến về 0 -> Loss bị triệt tiêu
        # pt càng gần 0 (đoán sai bét) -> (1-pt) tiến về 1 -> Giữ nguyên hình phạt
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [3]:
# ==========================================
# 3. HÀM KHỞI TẠO MODEL MỚI (Tránh rò rỉ bộ nhớ)
# ==========================================
def create_model():
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    num_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(num_features, 1)
    return model.to(CONFIG["device"])

In [4]:
# ==========================================
# 4. CHUẨN BỊ K-FOLD VÀ CHẠY HUẤN LUYỆN
# ==========================================
df = pd.read_csv(CONFIG["csv_path"])
test_df = pd.read_csv(CONFIG["test_csv"])
test_dataset = FloodDataset(test_df, CONFIG["test_img_dir"], "image_id", None, val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

skf = StratifiedKFold(n_splits=CONFIG["n_splits"], shuffle=True, random_state=CONFIG["seed"])
criterion = FocalLoss(alpha=0.6, gamma=2.0).to(CONFIG["device"])

# Mảng lưu trữ kết quả xác suất của tập Test từ 5 Folds
test_preds_all_folds = np.zeros(len(test_df))

print(f"=== BẮT ĐẦU HUẤN LUYỆN {CONFIG['n_splits']}-FOLD CROSS VALIDATION ===")

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label"])):
    print(f"\n--- ĐANG CHẠY FOLD {fold + 1}/{CONFIG['n_splits']} ---")
    
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_loader = DataLoader(FloodDataset(train_df, CONFIG["train_img_dir"], "id", "label", train_transform), batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(FloodDataset(val_df, CONFIG["train_img_dir"], "id", "label", val_test_transform), batch_size=CONFIG["batch_size"], shuffle=False)
    
    model = create_model()
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
    
    best_f1 = 0
    
    for epoch in range(CONFIG["epochs"]):
        # Train
        model.train()
        for images, labels in tqdm(train_loader, desc=f"Fold {fold+1} - Epoch {epoch+1}", leave=False):
            images, labels = images.to(CONFIG["device"]), labels.to(CONFIG["device"])
            optimizer.zero_grad()
            loss = criterion(model(images).squeeze(1), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
        # Validation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                outputs = model(images.to(CONFIG["device"])).squeeze(1)
                preds = (torch.sigmoid(outputs) >= 0.5).int().cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())
        
        val_f1 = f1_score(all_labels, all_preds, zero_division=0)
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f"best_model_fold_{fold}.pth")
            
    print(f">> Fold {fold+1} hoàn tất | Best Val F1: {best_f1:.4f}")
    
    # Sinh dự đoán trên tập Test cho Fold hiện tại (áp dụng luôn TTA)
    print(f"Đang dự đoán tập Test với model Fold {fold+1}...")
    model.load_state_dict(torch.load(f"best_model_fold_{fold}.pth"))
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(CONFIG["device"])
            probs_orig = torch.sigmoid(model(images).squeeze(1))
            probs_flipped = torch.sigmoid(model(torch.flip(images, dims=[3])).squeeze(1))
            probs = ((probs_orig + probs_flipped) / 2.0).cpu().numpy()
            fold_preds.extend(probs)
            
    # Cộng dồn xác suất vào mảng tổng
    test_preds_all_folds += np.array(fold_preds)

=== BẮT ĐẦU HUẤN LUYỆN 5-FOLD CROSS VALIDATION ===

--- ĐANG CHẠY FOLD 1/5 ---


>> Fold 1 hoàn tất | Best Val F1: 0.9231
Đang dự đoán tập Test với model Fold 1...


C:\Users\admin\AppData\Local\Temp\ipykernel_20720\4033473333.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"best_model_fold_{fold}.p


--- ĐANG CHẠY FOLD 2/5 ---


>> Fold 2 hoàn tất | Best Val F1: 0.9133
Đang dự đoán tập Test với model Fold 2...


...

>> Fold 4 hoàn tất | Best Val F1: 0.9199
Đang dự đoán tập Test với model Fold 4...


C:\Users\admin\AppData\Local\Temp\ipykernel_20720\4033473333.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"best_model_fold_{fold}.p


--- ĐANG CHẠY FOLD 5/5 ---


>> Fold 5 hoàn tất | Best Val F1: 0.9239
Đang dự đoán tập Test với model Fold 5...


C:\Users\admin\AppData\Local\Temp\ipykernel_20720\4033473333.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"best_model_fold_{fold}.p

In [5]:
# ==========================================
# 5. TỔNG HỢP KẾT QUẢ (ENSEMBLE) VÀ TẠO FILE NỘP
# ==========================================
print("\n=== ĐANG TỔNG HỢP KẾT QUẢ TỪ 5 FOLDS ===")
# Chia trung bình xác suất
test_preds_avg = test_preds_all_folds / CONFIG["n_splits"]

# Quyết định nhãn cuối cùng (ngưỡng 0.45 hoặc 0.5)
final_labels = (test_preds_avg >= 0.45).astype(int)

sub_df = pd.DataFrame({
    "id": test_df["image_id"],
    "prob": test_preds_avg
})

sub_df.to_csv("submission_convnext_ensemble.csv", index=False)
print("=== THÀNH CÔNG! Đã tạo file submission_convnext_ensemble.csv ===")


=== ĐANG TỔNG HỢP KẾT QUẢ TỪ 5 FOLDS ===
=== THÀNH CÔNG! Đã tạo file submission_convnext_ensemble.csv ===
